# Exercise 5: Dimensionality Reduction 1 — Spike Waveform Analysis

| | |
|---|---|
| **Topic** | PCA on spike waveforms: per-channel PCA, full-space PCA, GMM clustering, ICA bonus |
| **Data** | `spike_waveshapes.mat` (multichannel spike waveforms from silicon probe) |
| **Prereqs** | Linear algebra basics; PCA from lecture |
| **Time** | ~2 hours |

## Learning objectives

- Visualise multichannel spike waveforms
- Apply PCA per-channel and in full space
- Cluster spike types with Gaussian Mixture Models + BIC model selection
- Compare PCA and ICA on the same data


## Setup

In [ ]:
import numpy as np
import scipy.io as sio
from scipy.stats import zscore
from scipy import signal
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from itertools import combinations

from sklearn.decomposition import PCA, FastICA
from sklearn.mixture import GaussianMixture
from sklearn.manifold import Isomap, TSNE

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## Part 1: Spike Waveform Analysis

Load multichannel spike waveforms recorded from a silicon probe.
Each spike is a 2D array (channels x time samples).

In [ ]:
data = sio.loadmat('../../data/spectral/spike_waveshapes.mat')
spk = data['spk']            # (n_spikes, n_channels, n_samples)
clu = data['clu'].squeeze()   # (n_spikes,)
n_spikes, n_channels, n_samples = spk.shape
print(f"Shape: {spk.shape} -- {n_spikes} spikes, {n_channels} channels, {n_samples} samples")
print(f"Clusters: {np.unique(clu)} ({len(np.unique(clu))} unique)")

### Task 1a: Visualize Spike Waveshapes

**Think about it:** Each spike is stored as a matrix (channels x time). What are
useful ways to visualize these? How would you compare waveforms across clusters?

In [ ]:
# TODO: Visualize spike waveforms
# Suggestions:
# - Use imshow to show individual spikes as images (channels x time)
# - Overlay waveforms from the same cluster on each channel
# - Show the mean waveform per cluster

raise NotImplementedError("Your code here")

<details>
<summary><strong>Hint 1: Conceptual</strong></summary>

Spike waveshapes have two dimensions: channels (recording sites on the probe) and
time (the waveform at each site). `imshow` treats this as an image;
line plots show the temporal shape per channel.
</details>

<details>
<summary><strong>Hint 2: API</strong></summary>

Use `np.unique(clu)` for cluster IDs. Boolean indexing: `spk[clu == c, ch, :]`
selects all spikes from cluster `c` on channel `ch`. Use `.mean(axis=0)` for
the average waveform.
</details>

<details>
<summary><strong>Hint 3: Partial code</strong></summary>

```python
unique_clu = np.unique(clu)
fig, axes = plt.subplots(n_channels, min(6, len(unique_clu)),
                         figsize=(18, 2 * n_channels), sharex=True)
for j, c in enumerate(unique_clu[:6]):
    mask = clu == c
    for i in range(n_channels):
        axes[i, j].plot(spk[mask, i, :].T[:50], alpha=0.15, linewidth=0.5)
        axes[i, j].plot(spk[mask, i, :].mean(axis=0), 'k', linewidth=1.5)
```
</details>

### Task 1b: Per-Channel PCA

Perform PCA on each channel's waveforms separately.

**Think about it:** Why does PCA center the data automatically? What would happen
if the data had very different scales across features?

**Tasks:**
1. Fit PCA on each channel's (n_spikes x n_samples) matrix
2. Plot the eigenspectrum (fraction of variance explained) for all channels
3. Visualize the first 3 PC loadings (eigenvectors) per channel
4. Plot PC1 vs PC2 scatter per channel, colored by cluster
5. Make pairwise plots of PC1 scores across channel pairs

In [ ]:
# TODO: PCA per channel
# For each channel, extract spk[:, ch, :], fit PCA, store scores + loadings + var_ratio

raise NotImplementedError("Your code here")

<details>
<summary><strong>Hint 1: Why center before PCA?</strong></summary>

PCA finds directions of maximum variance. Without centering, the first PC points
toward the data centroid rather than capturing shape variation.
`sklearn.decomposition.PCA` centers automatically -- no manual step needed.
For spike waveforms (same units on all channels), centering suffices; scaling
is only needed when features have very different units.
</details>

<details>
<summary><strong>Hint 2: API</strong></summary>

```python
from sklearn.decomposition import PCA
pca = PCA()                         # keep all components
scores = pca.fit_transform(X)       # X: (n_spikes, n_samples)
pca.explained_variance_ratio_       # fraction of variance per PC
pca.components_                     # loadings (n_components, n_samples)
```
</details>

<details>
<summary><strong>Hint 3: Partial code</strong></summary>

```python
pca_ch = []
for ch in range(n_channels):
    pca = PCA()
    scores = pca.fit_transform(spk[:, ch, :])
    pca_ch.append({'scores': scores,
                   'loadings': pca.components_,
                   'var_ratio': pca.explained_variance_ratio_})

# Eigenspectrum
fig, ax = plt.subplots()
for ch in range(n_channels):
    ax.plot(pca_ch[ch]['var_ratio'], label=f'Ch {ch}')
ax.set_xlabel('PC'); ax.set_ylabel('Variance Explained')
ax.legend()
```
</details>

In [ ]:
# TODO: Plot eigenspectrum, loadings, and PC scatterplots
# - Eigenspectrum: var_ratio vs PC index for each channel
# - Loadings: first 3 PCs per channel as line plots
# - PC1 vs PC2 scatter per channel colored by cluster (clu)
# - Pairwise cross-channel PC1 scatterplots

raise NotImplementedError("Your code here")

In [ ]:
# --- Checkpoint 1b: per-channel PCA ---
# Verifies your Task 1b output. Expects `pca_ch` as a list of length
# n_channels, each entry having 'scores', 'loadings', 'var_ratio' keys.
# If you used a different variable name, rename or edit this cell.
try:
    assert isinstance(pca_ch, list), 'pca_ch must be a list'
    assert len(pca_ch) == n_channels, \
        f'pca_ch should have {n_channels} entries, got {len(pca_ch)}'
    for ch, item in enumerate(pca_ch):
        for k in ('scores', 'loadings', 'var_ratio'):
            assert k in item, f'pca_ch[{ch}] missing key {k!r}'
    print(f'\u2713 Checkpoint 1b passed: pca_ch has {len(pca_ch)} per-channel entries')
except NameError:
    print('\u26a0 `pca_ch` not defined yet \u2014 finish Task 1b then re-run this cell')

### Task 1c: Full-Space PCA

Reshape spike data from (n_spikes, n_channels, n_samples) to
(n_spikes, n_channels x n_samples) and apply PCA to the joint space.

**Think about it:** What information do the eigenvectors capture when reshaped
back to (n_channels, n_samples)?

**Tasks:**
1. Reshape and fit PCA
2. Plot cumulative eigenspectrum (also try log scale for eigenvalues)
3. Visualize first 16 eigenvectors reshaped as channel x time images
4. Pairwise scatterplots of first 6 PC scores, colored by cluster

In [ ]:
# TODO: Full-space PCA

raise NotImplementedError("Your code here")

<details>
<summary><strong>Hint 1: Reshaping</strong></summary>

`X_full = spk.reshape(n_spikes, -1)` flattens the channel and time dimensions.
Each row is now a vector of length n_channels x n_samples. After PCA, reshape
eigenvectors back: `pca.components_[i].reshape(n_channels, n_samples)`.
</details>

<details>
<summary><strong>Hint 2: Dual-axis eigenspectrum</strong></summary>

```python
fig, ax1 = plt.subplots()
ax1.plot(np.cumsum(pca.explained_variance_ratio_))
ax1.set_ylabel('Cumulative Variance')
ax2 = ax1.twinx()
ax2.plot(10 * np.log10(pca.explained_variance_), 'r--')
ax2.set_ylabel('Log Eigenvalue (dB)')
```
</details>

In [ ]:
# --- Checkpoint 1c: full-space PCA ---
# Verifies your Task 1c output. Expects `X_full` (flattened spikes),
# `pca_full` (the sklearn PCA object), and `scores_full` (projected scores).
try:
    expected_feats = n_channels * n_samples
    assert X_full.shape == (n_spikes, expected_feats), \
        f'X_full should have shape ({n_spikes}, {expected_feats}), got {X_full.shape}'
    assert scores_full.shape[0] == n_spikes, \
        f'scores_full should have {n_spikes} rows, got {scores_full.shape[0]}'
    assert hasattr(pca_full, 'components_'), 'pca_full should be a fitted sklearn PCA'
    print(f'\u2713 Checkpoint 1c passed: X_full {X_full.shape}, scores_full {scores_full.shape}')
except NameError as e:
    print(f'\u26a0 {e} \u2014 finish Task 1c then re-run this cell')

### Task 1d: GMM Clustering

Use Gaussian Mixture Models on PCA scores and select the number of
components via BIC (Bayesian Information Criterion).

**Think about it:** Why use PCA scores instead of raw waveforms for clustering?
How does BIC balance fit quality against model complexity?

In [ ]:
# TODO: GMM clustering with BIC model selection
# 1. Select first ~10 PCs from the full-space PCA
# 2. Fit GMM for k = 2..24, record BIC for each
# 3. Plot BIC vs k, find best k
# 4. Compare GMM labels to provided cluster labels in PC1-PC2 scatter

raise NotImplementedError("Your code here")

<details>
<summary><strong>Hint 1: BIC model selection</strong></summary>

BIC penalizes overly complex models. Lower BIC = better trade-off between
fit and complexity. Sweep over k values and pick the minimum.
</details>

<details>
<summary><strong>Hint 2: API</strong></summary>

```python
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
gmm.fit(X)
bic = gmm.bic(X)
labels = gmm.predict(X)
```
</details>

In [ ]:
# --- Checkpoint 1d: GMM clustering ---
# Verifies your Task 1d output. Expects `best_k` (the BIC-chosen number
# of mixture components) as an int in a plausible range.
try:
    assert isinstance(best_k, (int, np.integer)), f'best_k must be int, got {type(best_k)}'
    assert 2 <= int(best_k) <= 30, f'best_k={best_k} outside plausible range [2, 30]'
    print(f'\u2713 Checkpoint 1d passed: BIC-optimal k = {int(best_k)}')
except NameError:
    print('\u26a0 `best_k` not defined yet \u2014 finish Task 1d then re-run this cell')

### Task 1e: ICA Decomposition (Bonus)

Apply FastICA to the full-space spike data. Explore how mixing matrix columns
(reshaped to channel x time) relate to cluster-specific waveshapes.

**Think about it:** ICA has sign and permutation ambiguities -- what does
this mean for interpreting the results?

In [ ]:
# TODO: ICA on full-space spike waveforms (bonus)
# - Fit FastICA with ~10 components on X_full
# - Visualize mixing matrix columns reshaped as channel x time images
# - Plot IC activation distributions colored by cluster

raise NotImplementedError("Your code here")

<details>
<summary><strong>Hint: ICA non-uniqueness</strong></summary>

Unlike PCA (components ordered by variance), ICA sources have no natural ordering,
and their signs can flip between runs. IC1 in one run might be IC3 in another.
To match ICs to clusters, compare mixing matrix profiles to mean waveshapes per
cluster, or look at the IC activation distributions.
</details>

## Reflection

**Questions to consider:**
1. How many PCs are needed to capture most of the spike waveform variance?
2. What does the eigenspectrum tell you about the intrinsic dimensionality of spike shapes?
3. How did BIC help you choose the number of GMM components? Were the clusters biologically meaningful?
4. How does ICA differ from PCA on this data — what extra structure (if any) did it reveal?
